# If you run this notebook for the first time, please execute the following cell  to generate seq embedding firstly.

In [ ]:
import torch
import os
from tqdm import tqdm
from unimol_tools import UniMolRepr
clf = UniMolRepr(data_type='molecule')
# Note: if you want to generate embedding for KM or KKM, just replace the 'KCAT' word with 'KM' or 'KKM' within the command below.
!python ../Code/extract.py esm1v_t33_650M_UR90S_1 ../Data/KCAT/NewestFeature/seq_str.fasta ../Data/KCAT/NewestFeature/esm1v_t33_650M_UR90S_1_embeding_1280 --repr_layers 33 --include per_tok
base = os.path.join("../Data/KCAT/NewestFeature/", "esm1v_t33_650M_UR90S_1_embeding_1280/")
def change(index, layer):
    data = torch.load(base+f'{index}.pt')
    data = data['representations'][layer]
    torch.save(data, base+f'{index}.pt')
file_list = os.listdir(base)
length = len(file_list)
for index in tqdm(range(length)):
    change(index, 33)
def trans_smiles(index_smiles_path, store_path):
    smiles = torch.load(index_smiles_path)
    n = len(smiles)
    smiles_list = []
    for i in range(n):
        smiles_list.append(smiles[i])
    reprs = clf.get_repr(smiles_list)
    torch.save(torch.tensor(reprs), store_path)
trans_smiles(f"../Data/KCAT/NewestFeature/index_smiles", f"../Data/KCAT/NewestFeature/smiles_embedding")

## 5-fold cross-validation, and for the other two kinetics predictions, setting the 'ModelType' to '_KM_ ' or '_KKM_ ' 

In [ ]:
import sys
sys.path.append("../Code/")
from model import Predictor
import torch
import torch.utils.data as data
import numpy as np
import timeit
import torch.nn as nn
import torch.optim as optim
import os
from rdkit import RDLogger
from utils import train_fold, get_pair_info
RDLogger.DisableLog('rdApp.*')
torch.backends.cudnn.allow_tf32 = True
torch.backends.cuda.matmul.allow_tf32 = True
def train(args):
    train_fold(**args)
for i in range(5):
    train(
            {
                'Model':Predictor,
                'ModelType':'KCAT',
                'args':{'mol_in_dim':167, 'hidden_dim':512, 'protein_dim':1280, 'layer':10, 'dropout':0.5,'att_layer':10},
                'lr': 1e-3,
                'radius': 4,
                'setting': "KcatPredictor",
                'loss_fn':nn.MSELoss(),
                'optim':torch.optim.AdamW,
                'schedule':torch.optim.lr_scheduler.MultiStepLR,
                'schedule_args':{'milestones':[30, 50, 80], 'gamma':0.9},
                'batchsize':200,
                'Epoch':100,
                'train_info':'MACCSKeys-train-1',
                'embeding_path':'../Data/KCAT/NewestFeature/esm1v_t33_650M_UR90S_1_embeding_1280/',
                'smiles_path':'../Data/KCAT/NewestFeature/index_smiles',
                'device':torch.device('cuda:0'),
                'source_path':'../Code/model.py',
                'dataset_split_func':get_pair_info,
                'split_args':{'path':'../Data/KCAT/NewestFeature/'},
                'close_infer':True,
                'molType':'MACCSKeys',
                'fold':i
            }
    )

# Just to execute the cell and it will train a model for _kcat_ prediction, the results and weights are stored at Results/.And set the 'ModelType' to '_KM_ ' or '_KKM_ ' to get the other two kinetics predictions.
### Note: When 'ModelType' changes, don't forget to modify the path of the dataset

In [ ]:
import sys
sys.path.append("../Code/")
from model import Predictor
import torch
import torch.utils.data as data
import numpy as np
import timeit
import torch.nn as nn
import torch.optim as optim
import os
from rdkit import RDLogger
from utils import train, get_pair_info
RDLogger.DisableLog('rdApp.*')
torch.backends.cudnn.allow_tf32 = True
torch.backends.cuda.matmul.allow_tf32 = True
def train(args):
    train(**args)
train(
        {
            'Model':Predictor,
            'ModelType':'KCAT',
            'args':{'mol_in_dim':167, 'hidden_dim':512, 'protein_dim':1280, 'layer':10, 'dropout':0.5,'att_layer':10},
            'lr': 1e-3,
            'radius': 4,
            'setting': "KcatPredictor",
            'loss_fn':nn.MSELoss(),
            'optim':torch.optim.AdamW,
            'schedule':torch.optim.lr_scheduler.MultiStepLR,
            'schedule_args':{'milestones':[30, 50, 80], 'gamma':0.9},
            'batchsize':200,
            'Epoch':100,
            'train_info':'MACCSKeys-train-1',
            'embeding_path':'../Data/KCAT/NewestFeature/esm1v_t33_650M_UR90S_1_embeding_1280/',
            'smiles_path':'../Data/KCAT/NewestFeature/index_smiles',
            'device':torch.device('cuda:0'),
            'source_path':'../Code/model.py',
            'dataset_split_func':get_pair_info,
            'split_args':{'path':'../Data/KCAT/NewestFeature/'},
            'close_infer':True,
            'molType':'MACCSKeys'
        }
)